# CICIDS2017 Multi-Class Classification (Without Destination Port Leakage)

This notebook evaluates models after dropping `Destination Port` to prevent shortcut feature leakage.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Cleaning

In [ ]:
files = glob.glob(r'../Datasets/CICIDS/*.csv')
print(f"Found {len(files)} CSV files.")

df_list = []
for f in files:
    sub_df = pd.read_csv(f)
    df_list.append(sub_df)

df = pd.concat(df_list, ignore_index=True)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].astype(str).str.strip().str.encode('ascii', 'ignore').str.decode('utf-8')
df['Label'] = df['Label'].replace('', 'UNKNOWN')

# Drop non-predictive identifier columns AND Destination Port to prevent leakage
drop_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp', 'Destination Port']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Clean Inf/NaNs
df = df.replace([np.inf, -np.inf], np.nan).dropna()

print("Processed Dataset Shape:", df.shape)
print("\nTarget Class Distribution:\n", df['Label'].value_counts())

## 2. Preprocessing & Train-Test Split

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']

# Label Encode Target
target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

# Scale Numerical Features
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

## 3. Model Training & Comparison

In [ ]:
models = {
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost (Hist)": xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, tree_method='hist', objective='multi:softmax', num_class=len(target_encoder.classes_)),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=100, random_state=42),
    "Random Forest (Subsampled)": RandomForestClassifier(n_estimators=50, max_samples=0.2, random_state=42, n_jobs=-1)
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    w_f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    m_f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Weighted Precision': prec,
        'Weighted Recall': rec,
        'Weighted F1': w_f1,
        'Macro F1': m_f1,
        'Time (s)': elapsed
    })
    print(f"{name} - Acc: {acc:.4f}, Weighted F1: {w_f1:.4f}, Macro F1: {m_f1:.4f}")

results_df = pd.DataFrame(results)
display(results_df)

## 4. Visualizing Results

In [ ]:
plt.figure(figsize=(12, 6))
melted_df = results_df.melt(id_vars='Model', value_vars=['Accuracy', 'Weighted F1', 'Macro F1'], var_name='Metric', value_name='Score')
sns.barplot(data=melted_df, x='Model', y='Score', hue='Metric')
plt.title('CICIDS2017 Model Performance Without Destination Port Leakage')
plt.ylim(0.4, 1.0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()